# 02b: Find trading-day boundaries

**Owner:** C

**Purpose:** Test whether the 12-13 window cycle in volatility comes from trading days. Overnight gaps should appear as periodic spikes in the distance between consecutive price vectors. If they do, assign each window to a day and a slot. Boundaries are found from prices only, and volatility is used only afterward to check them.

**Input files:**
- `data/processed/tick_sizes.parquet` and `data/processed/time_order.parquet` (from `02_time_order`)
- `data/raw/train.csv` (validation only, read after `day_slots.parquet` is saved)

**Output files:**
- `data/processed/day_slots.parquet` (`time_id`, `day_id`, `slot`)
- `figures/consecutive_distance.png`
- `report/day_boundary_results.json`

**Limits:** `day_id` follows the recovered order, so it is not a calendar date. Days with fewer windows than usual cannot show which slot is missing.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

sys.path.append("..")

from src.day_slots import (
    assign_day_slots,
    boundary_scores,
    consecutive_distances,
    detect_boundary_windows,
    dominant_period,
    drop_false_boundaries,
    local_log_residuals,
    otsu_threshold,
    segment_length_counts,
)
from src.time_order import build_log_price_matrix, lag_autocorrelation, standardize_columns

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
FIGURES = Path("../figures")
REPORT = Path("../report")
SEED = 5650
N_NULL = 200
results = {"seed": SEED}

## Step 1: Distances along the recovered order

In [ ]:
ticks = pl.read_parquet(PROCESSED / "tick_sizes.parquet")
order = pl.read_parquet(PROCESSED / "time_order.parquet").sort("time_rank")
time_ids = np.sort(ticks["time_id"].unique().to_numpy())
log_price, stock_ids, _ = build_log_price_matrix(ticks, time_ids)
points = standardize_columns(log_price)
position = {time_id: i for i, time_id in enumerate(time_ids.tolist())}
ordered_time_ids = order["time_id"].to_numpy()
ordered_points = points[[position[t] for t in ordered_time_ids.tolist()]]

distances = consecutive_distances(ordered_points)
log_distances = np.log(distances)
period = dominant_period(log_distances)
acf_log_distance = [lag_autocorrelation(log_distances, lag) for lag in range(1, 41)]

results["distances"] = {
    "n": len(distances),
    "quantiles": {str(q): float(np.quantile(distances, q)) for q in (0, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1)},
    "dominant_period_windows": period,
    "implied_days": len(ordered_time_ids) / period,
    "acf_log_distance_lags_1_to_40": acf_log_distance,
}
print(json.dumps(results["distances"], indent=1))

## Step 2: Detect the spikes

The threshold comes from the data, using Otsu's method on a two-sided score. The rolling median window is three times the period found above, and the score adds the steps on both sides of each window. A simple threshold on single distances is included to show why that is not enough: spikes often come in adjacent pairs, which creates many false one-window segments.

To check that the segment lengths are not an accident of the method, the same threshold is also applied to 200 shuffled versions of the residuals.

In [ ]:
rolling_window = 2 * int(round(3 * period) // 2) + 1
residuals = local_log_residuals(distances, rolling_window)
scores = boundary_scores(residuals)
threshold = otsu_threshold(scores)
detected = detect_boundary_windows(scores, threshold)


def length_summary(boundaries):
    """Return count and share summaries of the segment lengths."""
    lengths = np.diff(boundaries)
    return {
        "boundaries": int(boundaries.size),
        "segments": int(lengths.size),
        "median": float(np.median(lengths)),
        "mean": float(lengths.mean()),
        "share_12_13": float(np.isin(lengths, [12, 13]).mean()),
        "share_13": float((lengths == 13).mean()),
        "share_11_14": float(np.isin(lengths, [11, 12, 13, 14]).mean()),
        "counts": segment_length_counts(boundaries),
    }


rng = np.random.default_rng(SEED)
null_shares = []
for _ in range(N_NULL):
    shuffled = boundary_scores(rng.permutation(residuals))
    null_shares.append(np.isin(np.diff(detect_boundary_windows(shuffled, threshold)), [12, 13]).mean())
null_shares = np.array(null_shares)

naive_threshold = otsu_threshold(log_distances)
naive_above = np.nonzero(log_distances > naive_threshold)[0]
naive_lengths = np.diff(naive_above)

score_median = np.median(scores)
score_mad = 1.4826 * np.median(np.abs(scores - score_median))
sensitivity = {}
for k in (1.0, 1.5, 2.0, 2.5, 3.0):
    sensitivity[str(k)] = length_summary(detect_boundary_windows(scores, score_median + k * score_mad))
    sensitivity[str(k)].pop("counts")
    sensitivity[str(k)]["threshold"] = float(score_median + k * score_mad)

results["detection"] = {
    "rolling_window": rolling_window,
    "otsu_threshold": threshold,
    "share_of_scores_above_threshold": float((scores > threshold).mean()),
    "two_sided_score": length_summary(detected),
    "null_share_12_13_mean": float(null_shares.mean()),
    "null_share_12_13_q025": float(np.quantile(null_shares, 0.025)),
    "null_share_12_13_q975": float(np.quantile(null_shares, 0.975)),
    "null_share_12_13_max": float(null_shares.max()),
    "n_null": N_NULL,
    "naive_single_distance": {
        "threshold": naive_threshold,
        "spikes": int(naive_above.size),
        "median_segment": float(np.median(naive_lengths)),
        "share_length_1": float((naive_lengths == 1).mean()),
        "share_12_13": float(np.isin(naive_lengths, [12, 13]).mean()),
    },
    "sensitivity_median_plus_k_mad": sensitivity,
}
print(json.dumps(results["detection"], indent=1))

## Step 3: Build days and slots

The hypothesis is supported if most detected segments are 12-13 windows long and shuffled data does not do the same. If so, each day is built as follows:

- **False alarms:** Two neighboring segments that would fit together within the typical length plus one window are merged, by removing the weaker boundary. Most short segments look like one day split in two.
- **Slots:** Slot 0 is the boundary window, the one between the two jumps. Slots count back along the recovered order from there.

These steps use the typical length, so the segment lengths after this cleaning cannot be used to test the hypothesis. That test uses the lengths above.

In [ ]:
detected_lengths = np.diff(detected)
hypothesis_supported = bool(
    np.isin(detected_lengths, [12, 13]).mean() > 0.5
    and np.isin(detected_lengths, [12, 13]).mean() > null_shares.max()
)
results["hypothesis_supported"] = hypothesis_supported
assert hypothesis_supported, "Detected segments do not support the day-boundary hypothesis"

values, counts = np.unique(detected_lengths, return_counts=True)
typical_length = int(values[np.argmax(counts)])
boundaries = drop_false_boundaries(detected, scores, max_length=typical_length + 1)
n_windows = len(ordered_time_ids)
day_id, slot = assign_day_slots(n_windows, boundaries, typical_length)

day_slots = pl.DataFrame(
    {"time_id": ordered_time_ids, "day_id": day_id, "slot": slot},
    schema={"time_id": pl.Int32, "day_id": pl.Int32, "slot": pl.Int32},
)
assert day_slots["time_id"].n_unique() == n_windows
assert day_slots["slot"].min() >= 0
day_slots.write_parquet(PROCESSED / "day_slots.parquet")

day_lengths = np.bincount(day_id)
day_values, day_counts = np.unique(day_lengths, return_counts=True)
slot_values, slot_counts = np.unique(slot, return_counts=True)
results["days"] = {
    "typical_length": typical_length,
    "max_merged_length": typical_length + 1,
    "boundaries_removed": int(detected.size - boundaries.size),
    "n_days": int(day_lengths.size),
    "windows_after_last_boundary": int(n_windows - 1 - boundaries[-1]),
    "day_length_counts": {int(v): int(c) for v, c in zip(day_values, day_counts)},
    "share_days_12_13": float(np.isin(day_lengths, [12, 13]).mean()),
    "share_windows_in_12_13_days": float(day_lengths[np.isin(day_lengths, [12, 13])].sum() / n_windows),
    "slot_counts": {int(v): int(c) for v, c in zip(slot_values, slot_counts)},
}
print(json.dumps(results["days"], indent=1))

## Step 4: Check the days against volatility

Volatility was not used to find the boundaries, so a strong slot pattern in volatility is independent evidence. The shape of the pattern also says something about direction. Markets are usually most volatile at the open and then calm down gradually, so a slow climb to a peak followed by a sudden drop would suggest that the recovered order runs backward in time. This is a hint, not proof.

In [ ]:
train = pl.read_csv(RAW / "train.csv").with_columns(pl.col("target").log().alias("log_target"))
market = train.group_by("time_id").agg(pl.col("log_target").mean().alias("m"))
m_by_time = dict(zip(market["time_id"].to_list(), market["m"].to_list()))
m_t = np.array([m_by_time[t] for t in ordered_time_ids.tolist()])


def variance_explained(values, groups):
    """Return the share of variance explained by group means."""
    total = np.sum((values - values.mean()) ** 2)
    within = sum(np.sum((values[groups == g] - values[groups == g].mean()) ** 2) for g in np.unique(groups))
    return float(1 - within / total)


m_day_demeaned = m_t - np.bincount(day_id, weights=m_t)[day_id] / np.bincount(day_id)[day_id]
position_forward = np.empty(n_windows, dtype=np.int64)
for d in np.unique(day_id):
    rows = np.nonzero(day_id == d)[0]
    position_forward[rows] = rows - rows[0]

profile = {int(s): float(m_day_demeaned[slot == s].mean()) for s in np.unique(slot)}

rng = np.random.default_rng(SEED)
placebo = [variance_explained(m_day_demeaned, rng.permutation(slot)) for _ in range(N_NULL)]

full_days = [d for d in np.unique(day_id) if np.bincount(day_id)[d] == typical_length]
overall = np.array([profile[s] for s in range(typical_length)])
agree_forward = 0
agree_reverse = 0
for d in full_days:
    rows = np.nonzero(day_id == d)[0]
    this_day = m_day_demeaned[rows][np.argsort(slot[rows])]
    others = (overall * len(full_days) - this_day) / (len(full_days) - 1)
    forward = np.corrcoef(this_day, others)[0, 1]
    reverse = np.corrcoef(this_day[::-1], others)[0, 1]
    agree_forward += forward > 0
    agree_reverse += reverse > forward

slot0_step = [
    m_day_demeaned[(day_id == d) & (slot == 0)].mean() - m_day_demeaned[(day_id == d) & (slot == 1)].mean()
    for d in full_days
]

results["validation"] = {
    "m_t_slot_variance_share_within_day": variance_explained(m_day_demeaned, slot),
    "m_t_position_forward_variance_share_within_day": variance_explained(m_day_demeaned, position_forward),
    "placebo_mean": float(np.mean(placebo)),
    "placebo_max": float(np.max(placebo)),
    "m_t_day_variance_share": variance_explained(m_t, day_id),
    "m_t_slot_variance_share_total": variance_explained(m_t, slot),
    "slot_profile_within_day": profile,
    "highest_slot": int(max(profile, key=profile.get)),
    "full_days": len(full_days),
    "share_full_days_matching_others": agree_forward / len(full_days),
    "share_full_days_better_reversed": agree_reverse / len(full_days),
    "share_full_days_slot0_above_slot1": float(np.mean(np.array(slot0_step) > 0)),
}
print(json.dumps(results["validation"], indent=1))

In [ ]:
PRIMARY, BASELINE, TRACE, GRID, TEXT = "#2a78d6", "#eb6834", "#b5b4ad", "#e8e7e2", "#52514e"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": TEXT, "axes.labelcolor": TEXT, "xtick.color": TEXT,
                     "ytick.color": TEXT, "axes.spines.top": False, "axes.spines.right": False})
fig = plt.figure(figsize=(10, 9), dpi=200, layout="constrained")
grid = fig.add_gridspec(3, 2)

ax = fig.add_subplot(grid[0, :])
ax.plot(np.arange(1, n_windows), distances, color=TRACE, lw=0.5)
ax.set(yscale="log", xlim=(0, n_windows), xlabel="Recovered time rank", ylabel="Distance (log scale)",
       title="Distance between consecutive price vectors, full sample")
ax.grid(axis="y", color=GRID)

zoom = 170
ax = fig.add_subplot(grid[1, :])
ax.plot(np.arange(1, zoom + 1), distances[:zoom], color=PRIMARY, lw=1.2, marker="o", ms=2.5, label="Distance")
for i, b in enumerate(boundaries[boundaries <= zoom]):
    ax.axvline(b, color=BASELINE, lw=1, ls="--", label="Detected boundary window" if i == 0 else None)
ax.set(xlim=(0, zoom), xlabel="Recovered time rank", ylabel="Distance",
       title=f"First {zoom} windows: spikes repeat about every {typical_length} windows")
ax.grid(axis="y", color=GRID)
ax.legend(frameon=False, loc="upper right", ncols=2)

ax = fig.add_subplot(grid[2, 0])
ax.hist(scores, bins=70, color=PRIMARY)
ax.axvline(threshold, color=BASELINE, lw=1.5, ls="--", label=f"Otsu threshold ({threshold:.2f})")
ax.set(xlabel="Two-sided spike score", ylabel="Windows", title="Spike scores and the threshold")
ax.legend(frameon=False)

ax = fig.add_subplot(grid[2, 1])
ax.bar(values, counts, color=PRIMARY, width=0.8)
ax.set(xlabel="Windows between detected spikes", ylabel="Segments", title="Segment lengths before cleaning",
       xticks=range(1, int(values.max()) + 1, 2))
fig.savefig(FIGURES / "consecutive_distance.png")
plt.show()

In [ ]:
(REPORT / "day_boundary_results.json").write_text(json.dumps(results, indent=2) + "\n")
print("Saved", REPORT / "day_boundary_results.json")